# 🚀 Full-Mission Rocket Lander — Falcon 9 + Mars EDL (Colab)

PPO agent learns a **complete rocket mission** in a pure-NumPy planar simulator with a real **TVC gimbal** (engine steers via torque), **fuel-optimal suicide burns**, and **Mars EDL** with a parachute that **rips** if deployed above 8 kPa dynamic pressure.

| Points | Event |
|---|---|
| ➕ | reference-descent tracking, nulling lateral offset |
| ➕ **+150 + 100 × fuel left** | safe landing (on pad, slow, upright) |
| ➖ **−150** | crash (miss pad / too fast / tip over) |
| ➖ **−230** | ripping the Mars chute (deployed too early) |

**⚠️ কপি-পেস্ট করবেন না!** GitHub-এর view থেকে কপি করলে নতুন লাইন হারিয়ে যায় (`import osos.chdir...` SyntaxError)।

**সবচেয়ে সহজ:** নিচের এক-ক্লিক লিংকে যান — যে ব্রাঞ্চ থেকে এই নোটবুক দেখছেন, Colab সরাসরি GitHub থেকে খুলবে:

👉 [(one-click Colab link — এই নোটবুকটির জন্য)](https://colab.research.google.com/github/MD-Mushfiqur123/rocket-lander/blob/vibe/full-mission-molab-03248f/colab_full_mission.ipynb)

ম্যানুয়াল পদ্ধতি: ফাইলটি **Download raw file** বাটন দিয়ে ডাউনলোড করুন → [colab.research.google.com](https://colab.research.google.com) → **File → Upload notebook**।

---
### সেলগুলো উপর থেকে ক্রমান্বয়ে চালান (Runtime → Run all)


In [ ]:
REPO = 'https://github.com/MD-Mushfiqur123/rocket-lander'
BRANCH = 'vibe/full-mission-molab-03248f'  # merge-এর পরে: 'main'
!git clone -q -b $BRANCH $REPO
print('cloned.')

In [ ]:
# PyBullet লাগবে না — pure NumPy + SB3, তাই install ~10 সেকেন্ড
!pip install -q gymnasium stable-baselines3
# legacy 'gym' package thakle SB3 ekta warning dey — amra Gymnasium use kori,
# tai stale gym uninstall kore dichhi (ar kichur jonno lagbe na):
!pip uninstall -y -q gym > /dev/null 2>&1
print('install done.')


In [ ]:
# 🧪 Physics self-check — scripted controllers দিয়ে প্রমাণ যে সিমুলেটর landable
# (landing: suicide burn 5x, ascent: 110 km gravity turn, mars: chute + retro 5x)
!cd /content/rocket-lander && PYTHONPATH=. python full_mission/scripted_test.py

In [ ]:
import os
os.chdir('/content/rocket-lander')
import shutil; shutil.rmtree('full_mission/__pycache__', ignore_errors=True)

# ⚙️ কনফিগারেশন:
PRESET      = 'earth_falcon9'   # 'earth_falcon9' বা 'mars_edl'
PHASE       = 'landing'         # earth: landing/reentry/boostback/ascent; mars: সবসময় reentry
TRAIN_STEPS = 1_000_000         # 500k quick; 1.5M+ full mission/robustness-এর জন্য
print('working dir:', os.getcwd())

In [ ]:
# 🚂 Training - curriculum randomization strength 0→1 প্রথম 60%
# প্রতি 100k স্টেপে checkpoint save + held-out eval
!cd /content/rocket-lander && PYTHONPATH=. python full_mission/train.py --preset {PRESET} --start-phase {PHASE} --steps {TRAIN_STEPS} --n-envs 4
MODEL_ZIP = f'{PRESET}_{PHASE}_ppo'  # train.py-এর default model name


In [ ]:
# 🎬 প্রশিক্ষিত AI-এর মিশন রিপ্লে — র্যান্ডমাইজড এপিসোড + GIF + স্কোর টেবিল
import base64, io, math, numpy as np
import matplotlib.figure
import matplotlib.patches as mpatches
from matplotlib.backends.backend_agg import FigureCanvasAgg
from PIL import Image
from stable_baselines3 import PPO
from full_mission.env import FullMissionEnv
from IPython.display import display, Markdown, HTML

import glob, os

_cand = f'{MODEL_ZIP}.zip'
if not os.path.exists(_cand):
    _zips = sorted(glob.glob('*_ppo*.zip'))
    if len(_zips) == 1:
        MODEL_ZIP = _zips[0].removesuffix('.zip')
        print(f'auto-detected model: {MODEL_ZIP}')
    else:
        raise FileNotFoundError(
            f'{_cand} pawajay nai (pawa zip: {_zips}) — '
            'Training cell ta age complete korte hobe! '
            'Runtime → Run all diye upor theke serially chalao, '
            'ba training cell (🚂) ta peek করে complete koro.'
        )

model = PPO.load(MODEL_ZIP)

def render(env, trail, thr, width=420, height=300):
    st = env.state
    pad_x = env._p['pad_x']
    x, z = st['pos'][0], max(st['pos'][1], 0.0)
    half = max(abs(x - pad_x) / 2 + 250.0, z * 1.25, 600.0)
    fig = matplotlib.figure.Figure(figsize=(width / 80, height / 80), dpi=80)
    fig.patch.set_facecolor('#0b0e1a')
    FigureCanvasAgg(fig)
    ax = fig.add_subplot(111)
    ax.set_facecolor('#0b0e1a')
    pad_hw = env.cfg['pad_half_width']
    ax.plot([-half, half], [0, 0], color='#2a2f45', lw=2)
    ax.add_patch(mpatches.Rectangle((pad_x - pad_hw, 0), 2 * pad_hw,
                 max(0.025 * half, 8.0), facecolor='#e6c700', zorder=2))
    if trail:
        tx, tz = zip(*trail)
        ax.plot(tx, tz, color='#4da6ff', lw=0.9, alpha=0.55)
    L = max(0.07 * half, 25.0); W = L * 0.22
    c, s = math.cos(st['theta']), math.sin(st['theta'])
    w = lambda px, pz: (x + px*c - pz*s, z + px*s + pz*c)
    ax.add_patch(mpatches.Polygon([w(-W,-L/2), w(W,-L/2), w(W,L/2), w(-W,L/2)],
                 closed=True, facecolor='#cc3333', edgecolor='#ff8080', zorder=5))
    if thr > 0.05:
        pl = L * (0.5 + 0.9 * thr)
        ax.add_patch(mpatches.Polygon([w(-W*0.7,-L/2), w(W*0.7,-L/2), w(0,-L/2-pl)],
                     closed=True, facecolor='#ffb347', alpha=0.9, zorder=4))
    if env.chute_deployed:
        cr = L * 1.6
        ax.add_patch(mpatches.Arc(w(0,L), 2*cr, 2*cr, theta1=20, theta2=160,
                     color='#dddddd', lw=2.5, zorder=3))
    hud = (f"alt {z:,.0f} m   speed {float(np.hypot(st['vel'][0], st['vel'][1])):.1f}"
           f" m/s   fuel {env._fuel_frac:.0%}")
    ax.text(0.02, 0.02, hud, transform=ax.transAxes, color='#9fb3c8', fontsize=6)
    ax.set_xlim(-half, half); ax.set_ylim(-0.06*half, 2*half)
    ax.set_aspect('equal'); ax.set_axis_off()
    fig.tight_layout(pad=0.2); fig.canvas.draw()
    return np.asarray(fig.canvas.buffer_rgba())[:, :, :3].astype(np.uint8)

hists, stats = [], []
for ep in range(3):
    env = FullMissionEnv(PRESET, start_phase=PHASE, seed=777_000 + ep)
    obs, _ = env.reset(); ep_r, done, hist = 0.0, False, []
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, r, term, trunc, info = env.step(action)
        ep_r += r; done = term or trunc
        st = env.state
        hist.append((float(st['pos'][0]), float(st['pos'][1]), st['theta'],
                     float(np.ravel(action)[1]), env.chute_deployed))
    env.close()
    hists.append(hist)
    stats.append((info['outcome'] or 'timeout', ep_r, info['speed'], info['tilt']))

frames = []
for hist in hists:
    k = max(1, len(hist) // 80)
    skel = FullMissionEnv(PRESET, start_phase=PHASE, seed=0); skel.reset(seed=0)
    trail = []
    for i in range(0, len(hist), k):
        px, pz, th, thr, chute = hist[i]
        trail.append((px, pz))
        skel.state = {'pos': np.array([px, pz]), 'vel': np.zeros(2),
                      'theta': th, 'omega': 0.0, 'fuel': skel._p['fuel0']}
        skel.chute_deployed = chute
        frames.append(render(skel, trail, thr))

pil = [Image.fromarray(f) for f in frames]
buf = io.BytesIO()
pil[0].save(buf, format='GIF', save_all=True, append_images=pil[1:],
            duration=90, loop=0)
html = '<img src="data:image/gif;base64,' + base64.b64encode(buf.getvalue()).decode() + '" width="420">'
rows = '\n'.join(
    f'| {i+1} | {st[0]} | {st[1]:.1f} | {st[2]:.2f} | {st[3]:.1f}° |'
    for i, st in enumerate(stats))
display(HTML(html))
display(Markdown('| Ep | Outcome | Reward | Touchdown speed | Final tilt |\n'
                 '|---|---|---|---|---|\n' + rows))


In [ ]:
# 💾 মডেল কে আপনার Google Drive-এ সেভ করুন (ঐচ্ছিক)
from google.colab import drive
drive.mount('/content/drive')
!cp {MODEL_ZIP}.zip /content/drive/MyDrive/
from google.colab import files
files.download(f'{MODEL_ZIP}.zip')

## 📌 Notes
- **Training time:** ~30–60 min for 1M steps প্রতি মিশনে (4 parallel envs)। Free CPU credentials → colab এ কাজ করে; GPU লাগে না (NumPy-based env, CPU-bound)।
- **Learning arc:** প্রথমে reference descent ট্র্যাক করা শেখে → তারপর soft touchdown → landing (1M এর আশপাশে প্রথম landing)। `ascent`/`boostback` ফেজে 1.5M+
- `mars_edl` চাইলে: config cell-এ `PRESET = 'mars_edl'` করে আবার ট্রেনিং চালান — chute timing নিজেই শিখবে
- Checkpoint প্রতি 100k স্টেপে সেভ হয় — মাঝপথেও থামিয়ে ডেমো দেখা যাবে
- Result রাখতে নোটবুক **download** করে রাখুন — সব কোড একসাথে!
